Import root folder as a path

In [35]:
import sys
sys.path.append('../')

Some SRC code imports

In [36]:
from models import load_model
from mydatasets import load_dataset

from pruning.heuristic_pruner import HeuristicPruner
from revival.affinity_revivor import AffinityRevivor
from core.token_manager import TokenManager

Pruner & Reviver

In [37]:
TOKENS_PER_BLOCK = {
    0.5: {
        "stepwise": [196,196,196,98,98,98,49,49,49,25,25,25],
        "linear": [196,180,165,149,134,118,103,87,72,56,41,25],
        "exponential": [196,163,135,112,93,77,64,53,44,36,30,25],
    },
    0.6: {
        "stepwise": [196,196,196,118,118,118,71,71,71,43,43,43],
        "linear": [196,182,168,154,140,126,113,99,85,71,57,43],
        "exponential": [196,171,149,130,113,98,86,75,65,57,49,43],
    },
    0.7: {
        "stepwise": [196,196,196,137,137,137,96,96,96,67,67,67],
        "linear": [196,184,173,161,149,137,126,114,102,90,79,67],
        "exponential": [196,178,161,146,133,120,109,99,90,81,74,67],
    }
}

def get_prune_and_revive_n_tokens(budget_target, pruning_schedule, keep_ratio):
    tokens_per_block_scheme = TOKENS_PER_BLOCK[budget_target][pruning_schedule]
    n_tokens_to_prune_per_block = [round(t * keep_ratio) for t in tokens_per_block_scheme]
    n_tokens_to_revive_per_block = [t - n for t, n in zip(tokens_per_block_scheme, n_tokens_to_prune_per_block)]
    return n_tokens_to_prune_per_block, n_tokens_to_revive_per_block

In [38]:
pruner = HeuristicPruner(criterion="C2")
revivor = AffinityRevivor(criterion="C4")
prune_n_tokens, revive_n_tokens = get_prune_and_revive_n_tokens(budget_target=0.7, pruning_schedule='exponential', keep_ratio=0.8)

token_manager = TokenManager(
    pruner=pruner,
    revivor=revivor,
    prune_ratio_or_n_tokens=prune_n_tokens,
    revive_ratio_or_n_tokens=revive_n_tokens,
)

Model

In [39]:
model = load_model(
    model_name="deit_small",
    pretrained=True,
    checkpoint_path="",
    token_manager=token_manager,
    can_tokens_revive=True,
).to("cuda")

Pesos pre-entrenados 'deit_small_patch16_224.fb_in1k' cargados desde timm


Get images

In [40]:
from pathlib import Path
from PIL import Image
from torchvision import transforms
import torch

In [41]:
def folder_images_to_tensor_batch(folder_path, batch_size=1):
    img_dir = Path(folder_path)
    img_paths = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}])[:batch_size]

    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])

    batch = torch.stack([preprocess(Image.open(p).convert("RGB")) for p in img_paths], dim=0)
    return batch

In [42]:
def get_imagenet_dataloader(data_dir, batch_size=1):
    datamodule = load_dataset(
        dataset_name="imagenet1k",
        data_dir=data_dir,
        batch_size=batch_size,
        image_size=224
    )

    datamodule.prepare_data()
    datamodule.setup()
    dataloader = datamodule.val_dataloader()

    return dataloader

Visualize tokens

In [43]:
import matplotlib as mpl

# To LaTeX
mpl.rcParams.update({
    "text.usetex": True,       # Usa LaTeX para todo el texto
    "font.family": "serif",    # Fuente tipo serif, igual que LaTeX
    "font.size": 22,         # Tamaño de fuente general
    "axes.labelsize": 22,      # Tamaño de etiquetas de ejes
    "xtick.labelsize": 8,      # Tamaño de ticks
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.titlesize": 12     # Tamaño del título
})

import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

from tqdm import tqdm

In [44]:
def plot_pruning_evolution(image_tensor, mask_list, patch_size=16, img_size=224, output_path="pruning_evolution.pdf"):
    """
    image_tensor: (B, 3, H, W)
    mask_list: List of 12 tensors of shape (B, num_tokens)
    batch_idx: index of the image in the batch to visualize
    """

    B = image_tensor.shape[0]
    NUM_PATCHES_SIDE = img_size // patch_size # 14 for 224x224 with 16x16 patches

    nrows, ncols = B, 12
    fig, axes = plt.subplots(
        nrows, ncols, 
        figsize=(ncols * 3, nrows * 3),
        gridspec_kw={'wspace': 0, 'hspace': 0}
    )

    for i, batch_idx in tqdm(enumerate(range(B)), total=B):
        # 1. Prepare the original image
        image = image_tensor[batch_idx].permute(1, 2, 0).cpu().numpy()  # (H, W, 3)
        image = (image - image.min()) / (image.max() - image.min())  # Normalize to [0, 1]

        for j, mask_block in enumerate(mask_list):
            # 2. Extract the mask for the current image and remove CLS token
            mask = mask_block[batch_idx]  # (num_tokens,)
            spatial_mask = mask[1:]

            # 3. Reshape the spatial mask to (num_patches_side, num_patches_side)
            spatial_mask = spatial_mask.view(NUM_PATCHES_SIDE, NUM_PATCHES_SIDE)

            # 4. Expand the mask to the original image size (H, W)
            mask_full = F.interpolate(
                spatial_mask.unsqueeze(0).unsqueeze(0),  # (1, 1, num_patches_side, num_patches_side)
                size=(img_size, img_size),  # (H, W)
                mode='nearest'
            ).squeeze().cpu().numpy()  # (H, W)

            # 5. Overlay white areas for pruned tokens using an alpha mask
            axes[i, j].imshow(image)
            overlay = np.zeros_like(image)  # White overlay
            alpha_mask = (mask_full == 1).astype(float) * 0.7
            axes[i, j].imshow(overlay, alpha=alpha_mask)
            axes[i, j].axis('off')

            if i == 0:
                axes[i, j].set_title(f"Block {j+1}", fontsize=50)

    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

    plt.savefig(output_path, bbox_inches="tight")
    # plt.show()
    plt.close()

# From folder
images_tensor = folder_images_to_tensor_batch("imgs", batch_size=10).to("cuda")
print("Image tensor shape from folder:", images_tensor.shape) # (B, 3, 224, 224)
mask_list = model.visualize_attention(images_tensor)[1:] # List of 12 tensors of shape (B, num_tokens)
print("Mask list len:", len(mask_list), "Mask shape example:", mask_list[0].shape) # (B, num_tokens)
plot_pruning_evolution(images_tensor, mask_list, patch_size=16, img_size=224, output_path="pruning_evolution_more_examples.pdf")
print("Visualization from folder images saved as 'pruning_evolution.pdf'\n")

# From imagenet
dataloader = get_imagenet_dataloader(data_dir="/home/Data/datasets/imagenet1k", batch_size=3)
inputs, _ = next(iter(dataloader))
print("Image tensor shape from ImageNet dataloader:", inputs.shape) # (B, 3, 224, 224)
inputs = inputs.to("cuda")
mask_list = model.visualize_attention(inputs)[1:] # List of 12 tensors of shape (B, num_tokens)
print("Mask list len:", len(mask_list), "Mask shape example:", mask_list[0].shape) # (B, num_tokens)
plot_pruning_evolution(inputs, mask_list, patch_size=16, img_size=224, output_path="pruning_evolution_imagenet.pdf")
print("Visualization from ImageNet dataloader saved as 'pruning_evolution_imagenet.pdf'")

Image tensor shape from folder: torch.Size([10, 3, 224, 224])
Mask list len: 12 Mask shape example: torch.Size([10, 197])


100%|██████████| 10/10 [00:00<00:00, 62.86it/s]


Visualization from folder images saved as 'pruning_evolution.pdf'

Image tensor shape from ImageNet dataloader: torch.Size([3, 3, 224, 224])
Mask list len: 12 Mask shape example: torch.Size([3, 197])


100%|██████████| 3/3 [00:00<00:00, 56.15it/s]


Visualization from ImageNet dataloader saved as 'pruning_evolution_imagenet.pdf'
